Before training, we applied three preprocessing steps to improve annotation consistency. First, we removed file-level annotations that did not provide temporal localization of signals within files. Second, we discarded KW annotations marked as uncertain when confidence labels were available. Third, we removed zero-duration annotations. These filtering steps removed only 1,532 annotations (0.74%), yielding a final dataset of 206,042 annotated events. DCLDE annotations distribution is shown in Table 1.

In [ ]:
import pandas as pd
df = pd.read_csv("/home/noah/HALLO_encoder_collection/scripts/DCLDE_2027/reproduce_k_palmer_2026_population_level/original_splits/annotations_w_calltype.csv")
# df = df.drop_duplicates(subset=common).reset_index(drop=True)

/tmp/ipykernel_197676/2358314874.py:2: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/noah/HALLO_encoder_collection/scripts/DCLDE_2027/reproduce_k_palmer_2026_population_level/original_splits/annotations_w_calltype.csv")


In [82]:

# check the duration and centertime vals to see if anything is missing
print(len(df[df['Duration'].isna()]))
print(len(df[df['CenterTime'].isna()]))

df.columns
# check the annotations where there is no actual duration specified. 
print(len(df[df['Duration'] == 0]))
print(len(df[df['FileBeginSec'] - df['FileEndSec'] == 0]))


0
0
650
650


we see there are 605 cases where there is no duration set for the annotation.

In [83]:
df['KW_certain'].value_counts()

KW_certain
1.0    57318
0.0      841
Name: count, dtype: int64

In [84]:
new_df = df[~(df['KW_certain'] == 0.0)]
print(len(new_df))
print(len(df))
print(len(df) - len(new_df))

new_df = new_df[new_df['Duration'] != 0.0]
print(len(new_df))
print(len(df) - len(new_df))

new_df = new_df[new_df['AnnotationLevel'] != 'File']
print(len(new_df))
print(len(df) - len(new_df))

206733
207574
841
206083
1491
206042
1532


In [85]:
print(df['AnnotationLevel'].value_counts())
df[df['AnnotationLevel'] == 'File']['Duration'].value_counts()
print(df[df['AnnotationLevel'] == 'File']['CenterTime'].value_counts())
df.columns


AnnotationLevel
Detection    169282
Call          37624
File            668
Name: count, dtype: int64
CenterTime
0.000000    627
1.225000     19
0.750000      2
0.560723      2
0.276476      1
0.817814      1
0.303060      1
0.717773      1
0.924334      1
0.320345      1
1.411621      1
1.098013      1
0.861283      1
1.439161      1
0.842187      1
0.574219      1
1.041250      1
1.540222      1
0.920258      1
2.210214      1
1.415897      1
0.973929      1
Name: count, dtype: int64


Index(['Soundfile', 'Dataset', 'LowFreqHz', 'HighFreqHz', 'FileEndSec', 'UTC',
       'FileBeginSec', 'ClassSpecies', 'KW', 'KW_certain', 'Ecotype',
       'Provider', 'AnnotationLevel', 'FilePath', 'FileOk', 'CallType',
       'CalltypeCategory', 'HasQ', 'CalltypeHasQ', 'LocalPath', 'LocalFileOk',
       'CenterTime', 'Duration', 'EcotypeCertain', 'Labels'],
      dtype='object')

The PP annotations resulted in slightly different results than the original annotaitons when running this removal logic, I need to pinpoint where this issue is... 

In [86]:
pp_df = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260827_090049/annotations.csv")
df.columns
keys = ['Soundfile', 'FileBeginSec', 'FileEndSec']

in_pp = pd.MultiIndex.from_frame(df[keys]).isin(pd.MultiIndex.from_frame(pp_df[keys]))
c = int(in_pp.sum())
print(f"{c} / {len(df)}")


/tmp/ipykernel_197676/3872199437.py:1: DtypeWarning: Columns (5,7,10,12,13,15,16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  pp_df = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260827_090049/annotations.csv")


207574 / 207574


In [87]:
new_pp_df = pp_df[~(pp_df['KW_certain'] == 0.0)]
print(len(new_pp_df))
print(len(pp_df))
print(len(pp_df) - len(new_pp_df))

new_pp_df = new_pp_df[new_pp_df['Duration'] != 0.0]
print(len(new_pp_df))
print(len(pp_df) - len(new_pp_df))

new_pp_df = new_pp_df[new_pp_df['AnnotationLevel'] != 'File']
print(len(new_pp_df))
print(len(pp_df) - len(new_pp_df))

1179346
1180187
841
1178741
1446
1178701
1486


We see here that there are less clips where duration = 0.0

In [88]:
removed_pp_df = pp_df[(pp_df['KW_certain'] == 0.0) | (pp_df['Duration'] == 0.0) | (pp_df['AnnotationLevel'] == 'File')]
print(len(removed_pp_df))

removed_df = df[(df['KW_certain'] == 0.0) | (df['Duration'] == 0.0) | (df['AnnotationLevel'] == 'File')]
print(len(removed_df))
print(len(removed_df) - len(removed_pp_df))



1486
1532
46


In [89]:
common = ['Soundfile', 'FileBeginSec', 'FileEndSec']

def summary(name, d):
    print(f"{name:14} rows={len(d):5}  distinct_keys={d[common].drop_duplicates().shape[0]:5}  "
          f"dup_extra={d.duplicated(subset=common).sum():4}  "
          f"key_nans={d[common].isna().any(axis=1).sum():4}")

summary("removed_df", removed_df)
summary("removed_pp_df", removed_pp_df)

# keys on one side but not the other
only_df = removed_df.merge(removed_pp_df[common].drop_duplicates(), on=common, how='left', indicator=True).query('_merge=="left_only"')
only_pp = removed_pp_df.merge(removed_df[common].drop_duplicates(), on=common, how='left', indicator=True).query('_merge=="left_only"')
print("only in removed_df   :", len(only_df))
print("only in removed_pp_df :", len(only_pp))

# which filter clause fires, each side
for name, d in [("df", removed_df), ("pp", removed_pp_df)]:
    print(name, "KW0:", (d['KW_certain']==0.0).sum(),
               "Dur0:", (d['Duration']==0.0).sum(),
               "File:", (d['AnnotationLevel']=='File').sum())

removed_df     rows= 1532  distinct_keys= 1472  dup_extra=  60  key_nans=   0
removed_pp_df  rows= 1486  distinct_keys= 1472  dup_extra=  14  key_nans=   0
only in removed_df   : 0
only in removed_pp_df : 0
df KW0: 841 Dur0: 650 File: 668
pp KW0: 841 Dur0: 605 File: 622


so there are 14 dupes in the pp df and 60 in the removed df

In [90]:
# full rows behind the 46-row gap: keys where removed_df has more copies than removed_pp_df
dc = removed_df.groupby(common).size()
pc = removed_pp_df.groupby(common).size()
gap = (dc - pc.reindex(dc.index, fill_value=0)).sort_values(ascending=False)
gap = gap[gap > 0]
print(f"{len(gap)} keys account for {gap.sum()} extra rows")

dup_keys = gap.index.to_frame(index=False)
dup_rows_df = removed_df.merge(dup_keys, on=common).sort_values(common)
dup_rows_pp = removed_pp_df.merge(dup_keys, on=common).sort_values(common)

# are the df copies byte-identical? if so, drop_duplicates reconciles the counts
print("exact full-row dups in removed_df:", removed_df.duplicated(keep=False).sum())
print("removed_df.drop_duplicates(subset=common):", removed_df.drop_duplicates(subset=common).shape[0])

pd.set_option('display.max_columns', None, 'display.width', None)
dup_rows_df


46 keys account for 46 extra rows
exact full-row dups in removed_df: 92
removed_df.drop_duplicates(subset=common): 1472


,Soundfile,Dataset,LowFreqHz,HighFreqHz,FileEndSec,UTC,FileBeginSec,ClassSpecies,KW,KW_certain,Ecotype,Provider,AnnotationLevel,FilePath,FileOk,CallType,CalltypeCategory,HasQ,CalltypeHasQ,LocalPath,LocalFileOk,CenterTime,Duration,EcotypeCertain,Labels
0,1562337136_0005.wav,orcasound_lab,NaN,NaN,1.121445,2019-07-05 14:32:16,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/orcasound_lab/1562337...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.560723,1.121445,False,Background
1,1562337136_0005.wav,orcasound_lab,NaN,NaN,1.121445,2019-07-05 14:32:16,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/orcasound_lab/1562337...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.560723,1.121445,False,Background
76,rpi-bush-point_2020_09_28_00_00_00.wav,bush_point,NaN,NaN,0.000000,2020-09-28 07:00:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/bush_point/rpi-bush-p...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
90,rpi-bush-point_2020_09_28_00_00_00.wav,bush_point,NaN,NaN,0.000000,2020-09-28 07:00:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/bush_point/rpi-bush-p...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
69,rpi-bush-point_2020_09_28_00_01_00.wav,bush_point,NaN,NaN,0.000000,2020-09-28 07:01:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/bush_point/rpi-bush-p...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,rpi-port-townsend_2020_09_29_19_42_00.wav,port_townsend,NaN,NaN,0.000000,2020-09-30 02:42:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/port_townsend/rpi-por...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
15,rpi-port-townsend_2020_09_29_19_43_00.wav,port_townsend,NaN,NaN,0.000000,2020-09-30 02:43:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/port_townsend/rpi-por...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
45,rpi-port-townsend_2020_09_29_19_43_00.wav,port_townsend,NaN,NaN,0.000000,2020-09-30 02:43:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/port_townsend/rpi-por...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background
23,rpi-port-townsend_2020_09_29_19_44_00.wav,port_townsend,NaN,NaN,0.000000,2020-09-30 02:44:00,0.0,AB,0,NaN,NaN,OrcaSound,File,E:/DCLDE/OrcaSound/Audio/port_townsend/rpi-por...,True,NaN,NaN,False,False,/data/DCLDE_2027/dclde_2027_killer_whales/orca...,True,0.000000,0.000000,False,Background


In [102]:
pp_df = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260827_090049/annotations.csv")
df = pd.read_csv("/home/noah/HALLO_encoder_collection/scripts/DCLDE_2027/reproduce_k_palmer_2026_population_level/original_splits/annotations_w_calltype.csv")


dropped_df = df.drop_duplicates().reset_index(drop=True)
dropped_pp_df = pp_df.drop_duplicates().reset_index(drop=True)

print(len(df), len(pp_df[pp_df['Labels'] != 'Background']))
print(len(dropped_df), len(dropped_pp_df[dropped_pp_df['Labels'] != 'Background']))

/tmp/ipykernel_197676/3982419962.py:1: DtypeWarning: Columns (5,7,10,12,13,15,16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  pp_df = pd.read_csv("/home/noah/HALLO_encoder_collection/data_raw/DCLDE_2027/20260827_090049/annotations.csv")
/tmp/ipykernel_197676/3982419962.py:2: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/noah/HALLO_encoder_collection/scripts/DCLDE_2027/reproduce_k_palmer_2026_population_level/original_splits/annotations_w_calltype.csv")


207574 207509
207509 207509


This confirms that the original annotations does have duplicates. So I will be sure to also include deduping in the set of preprocessing steps.